### Final ORCA Temporal XGBoost Model

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

#### Loading and konowing the Dataset

In [3]:
df = pd.read_csv("../data/marine_risk_processed.csv")

print("Dataset shape:", df.shape)

df["datetime"] = pd.to_datetime(df["datetime"])

print("Date range:")
print("Start:", df["datetime"].min())
print("End:", df["datetime"].max())

df.head()

Dataset shape: (4028, 14)
Date range:
Start: 2020-01-01 12:00:00
End: 2026-08-18 12:00:00


,datetime,wind_speed_kts,wind_direction_deg,air_temperature_c,water_temperature_c,relative_humidity_pct,air_pressure_hpa,station_name,latitude,longitude,risk_class,risk_label,wave_height_m,wave_period_s
0,2020-01-01 12:00:00,NaN,NaN,25.3,NaN,66.6,1014.5,arabian_sea_15n65e,15.0,65.0,0,LOW,0.843357,6.377153
1,2020-01-02 12:00:00,NaN,NaN,25.2,NaN,66.3,1016.1,arabian_sea_15n65e,15.0,65.0,0,LOW,0.845298,5.555953
2,2020-01-03 12:00:00,NaN,NaN,25.2,NaN,65.2,1015.8,arabian_sea_15n65e,15.0,65.0,0,LOW,1.238599,5.327560
3,2020-01-04 12:00:00,NaN,NaN,25.1,NaN,64.2,1014.3,arabian_sea_15n65e,15.0,65.0,0,LOW,1.245345,5.690114
4,2020-01-05 12:00:00,NaN,NaN,25.1,NaN,59.8,1012.8,arabian_sea_15n65e,15.0,65.0,0,LOW,1.034001,6.324389


In [4]:
df["month"] = df["datetime"].dt.month
df["hour"] = df["datetime"].dt.hour

#### Defining Exact Features and Targets

In [5]:
features = [
    "wind_speed_kts",
    "wind_direction_deg",
    "wave_height_m",
    "wave_period_s",
    "air_temperature_c",
    "water_temperature_c",
    "relative_humidity_pct",
    "air_pressure_hpa",
    "latitude",
    "longitude",
    "month",
    "hour"
]

target = "risk_class"

X = df[features]
y = df[target]

print("Features:")
print(features)

print("\nClass distribution:")
print(y.value_counts().sort_index())

Features:
['wind_speed_kts', 'wind_direction_deg', 'wave_height_m', 'wave_period_s', 'air_temperature_c', 'water_temperature_c', 'relative_humidity_pct', 'air_pressure_hpa', 'latitude', 'longitude', 'month', 'hour']

Class distribution:
risk_class
0    1967
1    1840
2     191
3      30
Name: count, dtype: int64


#### Checking Missing Values

In [6]:
print(X.isnull().sum())

wind_speed_kts            294
wind_direction_deg        294
wave_height_m               0
wave_period_s               0
air_temperature_c          10
water_temperature_c       917
relative_humidity_pct      86
air_pressure_hpa         2647
latitude                    0
longitude                   0
month                       0
hour                        0
dtype: int64


#### Performing Temporal Split for Training and Testing the Data

In [7]:
split_date = pd.Timestamp("2024-01-01")

train_mask = df["datetime"] < split_date
test_mask = df["datetime"] >= split_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Training period:")
print(df.loc[train_mask, "datetime"].min(), "to",
      df.loc[train_mask, "datetime"].max())

print("\nTest period:")
print(df.loc[test_mask, "datetime"].min(), "to",
      df.loc[test_mask, "datetime"].max())

print("\nTraining samples:", len(X_train))
print("Test samples:", len(X_test))

Training period:
2020-01-01 12:00:00 to 2021-10-22 12:00:00

Test period:
2024-08-09 12:00:00 to 2026-08-18 12:00:00

Training samples: 2375
Test samples: 1653


#### Checking Training and Test Class Distribution

In [8]:
print("Training class distribution:")
print(y_train.value_counts().sort_index())

print("\nTest class distribution:")
print(y_test.value_counts().sort_index())

Training class distribution:
risk_class
0    1189
1    1048
2     111
3      27
Name: count, dtype: int64

Test class distribution:
risk_class
0    778
1    792
2     80
3      3
Name: count, dtype: int64


#### Calculate Balanced Class Weights

In [9]:
classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weight_dict = dict(
    zip(classes, class_weights)
)

print("Class weights:")
print(class_weight_dict)

Class weights:
{0: 0.49936921783010935, 1: 0.5665553435114504, 2: 5.349099099099099, 3: 21.99074074074074}


In [10]:
sample_weights = y_train.map(class_weight_dict).to_numpy()

print("\nFirst 10 sample weights:")
print(sample_weights[:10])


First 10 sample weights:
[0.49936922 0.49936922 0.49936922 0.49936922 0.49936922 0.49936922
 0.49936922 0.49936922 0.49936922 0.49936922]


#### Configuring XGBoost Model

In [11]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=4,

    n_estimators=300,
    learning_rate=0.05,

    max_depth=5,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.5,

    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

#### Training using Sample Weights

In [12]:
print("Training ORCA XGBoost model...")

model.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

print("Training completed!")

Training ORCA XGBoost model...
Training completed!


#### Evaluating both training and temporal test data

In [13]:
def evaluate_model(model, X_data, y_true, dataset_name):
    
    y_pred = model.predict(X_data)
    
    accuracy = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )
    
    print("=" * 50)
    print(dataset_name)
    print("=" * 50)
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    
    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2, 3],
            target_names=[
                "LOW",
                "MODERATE",
                "HIGH",
                "EXTREME"
            ],
            zero_division=0
        )
    )
    
    print("\nConfusion Matrix:")
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3]
    )
    
    print(cm)
    
    return {
        "accuracy": float(accuracy),
        "macro_f1": float(macro_f1),
        "confusion_matrix": cm.tolist()
    }

In [14]:
train_metrics = evaluate_model(
    model,
    X_train,
    y_train,
    "TRAINING RESULTS"
)

test_metrics = evaluate_model(
    model,
    X_test,
    y_test,
    "TEMPORAL HOLDOUT TEST RESULTS"
)

TRAINING RESULTS
Accuracy: 1.0000
Macro F1: 1.0000

Classification Report:
              precision    recall  f1-score   support

         LOW       1.00      1.00      1.00      1189
    MODERATE       1.00      1.00      1.00      1048
        HIGH       1.00      1.00      1.00       111
     EXTREME       1.00      1.00      1.00        27

    accuracy                           1.00      2375
   macro avg       1.00      1.00      1.00      2375
weighted avg       1.00      1.00      1.00      2375


Confusion Matrix:
[[1189    0    0    0]
 [   0 1048    0    0]
 [   0    0  111    0]
 [   0    0    0   27]]
TEMPORAL HOLDOUT TEST RESULTS
Accuracy: 0.9982
Macro F1: 0.9990

Classification Report:
              precision    recall  f1-score   support

         LOW       1.00      1.00      1.00       778
    MODERATE       1.00      1.00      1.00       792
        HIGH       1.00      1.00      1.00        80
     EXTREME       1.00      1.00      1.00         3

    accuracy      

### Ablation/Single Feature Diagnostic
<p>Checking how much prediction can be made with wave_height alone</p>

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Only use wave height
X_wave = df[["wave_height_m"]]
y_wave = df["risk_class"]

# Same temporal split
wave_train_mask = df["datetime"] < pd.Timestamp("2024-01-01")
wave_test_mask = df["datetime"] >= pd.Timestamp("2024-01-01")

X_wave_train = X_wave.loc[wave_train_mask]
X_wave_test = X_wave.loc[wave_test_mask]

y_wave_train = y_wave.loc[wave_train_mask]
y_wave_test = y_wave.loc[wave_test_mask]

# Small decision tree
wave_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

wave_model.fit(
    X_wave_train,
    y_wave_train
)

wave_pred = wave_model.predict(X_wave_test)

print(
    "Wave-height-only accuracy:",
    accuracy_score(y_wave_test, wave_pred)
)

print(
    classification_report(
        y_wave_test,
        wave_pred,
        labels=[0, 1, 2, 3],
        target_names=[
            "LOW",
            "MODERATE",
            "HIGH",
            "EXTREME"
        ],
        zero_division=0
    )
)

Wave-height-only accuracy: 0.9751966122202057
              precision    recall  f1-score   support

         LOW       0.95      1.00      0.97       778
    MODERATE       1.00      0.95      0.97       792
        HIGH       1.00      1.00      1.00        80
     EXTREME       1.00      1.00      1.00         3

    accuracy                           0.98      1653
   macro avg       0.99      0.99      0.99      1653
weighted avg       0.98      0.98      0.98      1653



In [16]:
pd.crosstab(
    pd.cut(
        df["wave_height_m"],
        bins=[-np.inf, 1.5, 2.2, 4.0, np.inf]
    ),
    df["risk_class"],
    normalize="index"
).round(3)

risk_class,0,1,2,3
wave_height_m,,,,
"(-inf, 1.5]",0.959,0.041,0.000,0.0
"(1.5, 2.2]",0.000,0.999,0.001,0.0
"(2.2, 4.0]",0.000,0.727,0.273,0.0
"(4.0, inf]",0.000,0.000,0.000,1.0


### Saving the Model

In [18]:
import os

os.makedirs("../models", exist_ok=True)

model.save_model(
    "../models/orca_xgb_model.json"
)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
import os

path = "../models/orca_xgb_model.json"

size_mb = os.path.getsize(path) / (1024 * 1024)

print(f"Model size: {size_mb:.2f} MB")

Model size: 1.58 MB
